# 🚀 GAFIME Starter Guide
**GPU-Accelerated Feature Interaction Mining Engine**

GAFIME automatically discovers feature interactions that matter for your target variable. Think of it as automated feature engineering — it systematically tests millions of feature combinations (like `log(f1) * sqrt(f2)`) and tells you which ones are predictive.

---

### What You'll Learn
1. **Quick Start** — Run GAFIME in 3 lines of code
2. **Understanding Results** — Read the diagnostic report
3. **Configuration** — Tune the search budget
4. **Low-Level API** — Use StaticBucket for maximum control
5. **Real Data** — Load from CSV/Parquet with GafimeStreamer

## 📦 Installation

```bash
# Clone and install
git clone https://github.com/onlyxItachi/GAFIME.git
cd GAFIME
pip install -e .

# (Optional) Build native backends for GPU acceleration
python setup.py build_ext --inplace
```

In [ ]:
# Verify installation
import gafime
print(f"GAFIME v{gafime.__version__}")

---
## 1️⃣ Quick Start — 3 Lines to Feature Discovery

GAFIME's high-level API is dead simple:
- **Input**: Your feature matrix `X` and target `y`
- **Output**: A `DiagnosticReport` telling you which interactions matter

In [ ]:
import numpy as np
from gafime import GafimeEngine, EngineConfig, ComputeBudget

# --- Generate synthetic data with KNOWN signals ---
np.random.seed(42)
n_samples = 5000

# 5 random features
f0 = np.random.randn(n_samples).astype(np.float32)
f1 = np.random.randn(n_samples).astype(np.float32)
f2 = np.random.randn(n_samples).astype(np.float32)
f3 = np.random.randn(n_samples).astype(np.float32)
f4 = np.random.randn(n_samples).astype(np.float32)  # Pure noise

# Target with planted signals
y = (
    0.7 * (f0 * f1) +                                    # Strong multiplicative
    0.3 * (np.log(np.abs(f2) + 1e-8) + f3) +            # Log-additive
    0.1 * np.random.randn(n_samples)                     # Noise
).astype(np.float32)

X = np.column_stack([f0, f1, f2, f3, f4])
feature_names = ["price", "volume", "volatility", "momentum", "noise"]

print(f"Dataset: {X.shape[0]} samples × {X.shape[1]} features")
print(f"Features: {feature_names}")
print(f"Planted signals: price*volume (strong), log(volatility)+momentum (moderate)")

In [ ]:
# --- Run GAFIME (the magic 3 lines) ---
engine = GafimeEngine()         # Uses default config
report = engine.analyze(X, y, feature_names=feature_names)

# Did GAFIME find a signal?
print(f"\n🎯 Decision: {report.decision.message}")
print(f"   Signal detected: {report.decision.signal_detected}")
print(f"   Backend used: {report.backend.name if report.backend else 'numpy'}")

---
## 2️⃣ Understanding the Diagnostic Report

The `DiagnosticReport` contains:

| Field | What it tells you |
|---|---|
| `interactions` | Every feature combo tested + its metric scores |
| `stability` | How stable each score is across resamples |
| `permutations` | p-values — is the signal real or random? |
| `decision` | Final verdict: signal detected or not |
| `warnings` | Any budget/memory/backend issues |

In [ ]:
# --- Explore the interactions ---
print("📊 Top Feature Interactions (sorted by Pearson |r|)\n")
print(f"{'Rank':<5} {'Features':<35} {'Pearson':>8} {'Spearman':>9} {'R²':>8}")
print("-" * 70)

# Sort interactions by absolute Pearson correlation
sorted_interactions = sorted(
    report.interactions,
    key=lambda x: abs(x.metrics.get('pearson', 0)),
    reverse=True
)

for rank, result in enumerate(sorted_interactions[:15], 1):
    names = " × ".join(result.feature_names)
    pearson = result.metrics.get('pearson', 0)
    spearman = result.metrics.get('spearman', 0)
    r2 = result.metrics.get('r2', 0)
    
    marker = " 🎯" if set(result.feature_names) & {"price", "volume", "volatility", "momentum"} else ""
    print(f"{rank:<5} {names:<35} {pearson:>8.4f} {spearman:>9.4f} {r2:>8.4f}{marker}")

In [ ]:
# --- Check stability: are the scores consistent? ---
print("📐 Score Stability (lower std = more reliable)\n")

for stab in report.stability[:10]:
    combo_names = [report.feature_names[i] for i in stab.combo]
    name = " × ".join(combo_names)
    
    for metric, mean_val in stab.metrics_mean.items():
        std_val = stab.metrics_std.get(metric, 0)
        status = "✅ Stable" if std_val < 0.10 else "⚠️ Unstable"
        print(f"  {name:<30} {metric}: {mean_val:.4f} ± {std_val:.4f}  {status}")

In [ ]:
# --- Check permutation tests: is it real or random? ---
print("🎲 Permutation Test p-values (lower = more significant)\n")

for perm in report.permutations[:10]:
    combo_names = [report.feature_names[i] for i in perm.combo]
    name = " × ".join(combo_names)
    
    for metric, p_val in perm.p_values.items():
        status = "✅ Significant" if p_val <= 0.05 else "❌ Not significant"
        print(f"  {name:<30} {metric}: p={p_val:.4f}  {status}")

In [ ]:
# --- Warnings from the engine ---
if report.warnings:
    print("⚠️ Warnings:")
    for w in report.warnings:
        print(f"  - {w}")
else:
    print("✅ No warnings — everything ran smoothly!")

---
## 3️⃣ Configuration — Tune the Search

GAFIME's behavior is controlled by two dataclasses:

### `ComputeBudget` — How much to search
```python
@dataclass(frozen=True)
class ComputeBudget:
    max_comb_size: int = 2        # Max features per interaction (2=pairwise)
    max_combinations_per_k: int = 5000  # Max combos to try per arity
    top_features_for_higher_k: int = 50 # Keep top N for arity 3+
    vram_budget_mb: int = 6144    # GPU memory budget
```

### `EngineConfig` — How to evaluate
```python
@dataclass(frozen=True)
class EngineConfig:
    budget: ComputeBudget              # Search budget
    metric_names: Tuple[str, ...]      # Metrics to compute
    backend: str = "auto"              # "auto", "cuda", "metal", "cpu", "numpy"
    permutation_tests: int = 25        # Number of permutation tests
    permutation_p_threshold: float = 0.05  # Significance threshold
    random_seed: Optional[int] = 7     # For reproducibility
```

In [ ]:
# --- Example: Deeper search with 3-way interactions ---

config = EngineConfig(
    budget=ComputeBudget(
        max_comb_size=3,              # Search up to 3-way interactions!
        max_combinations_per_k=2000,  # Limit combos for speed
        top_features_for_higher_k=5,  # Only top 5 features go to arity 3
    ),
    metric_names=("pearson", "spearman"),  # Only compute these metrics
    backend="auto",                   # Let GAFIME pick the best backend
    permutation_tests=10,             # Fewer permutation tests (faster)
    random_seed=42,
)

engine = GafimeEngine(config)
report_deep = engine.analyze(X, y, feature_names=feature_names)

print(f"Decision: {report_deep.decision.message}")
print(f"Interactions found: {len(report_deep.interactions)}")
print(f"Warnings: {len(report_deep.warnings)}")

In [ ]:
# --- Example: Force a specific backend ---

# Force NumPy backend (always available, no compilation needed)
config_numpy = EngineConfig(backend="numpy")
engine_numpy = GafimeEngine(config_numpy)

report_numpy = engine_numpy.analyze(X, y, feature_names=feature_names)
print(f"NumPy backend: {report_numpy.backend.name if report_numpy.backend else 'numpy'}")
print(f"Decision: {report_numpy.decision.message}")

# Available backends: "auto", "cuda", "metal", "cpu", "core", "numpy"

---
## 4️⃣ Low-Level API — StaticBucket (Advanced)

For maximum control, use the `StaticBucket` API directly. This is what GAFIME uses internally — it goes straight to the GPU kernel.

**When to use this:**
- Custom operator combinations (log, sqrt, exp, tanh on each feature)
- Custom interaction types (multiply, add, divide, subtract)
- Cross-validation with fold masks
- Benchmarking kernel throughput

> ⚠️ This API requires a compiled native backend (CUDA or CPU). If you only have NumPy, use the high-level `GafimeEngine` instead.

In [ ]:
import time
from itertools import combinations

try:
    from gafime.backends.fused_kernel import (
        StaticBucket,
        UnaryOp,
        InteractionType,
        compute_pearson_from_stats,
        create_fold_mask,
    )
    HAS_NATIVE = True
    print("✅ Native backend available — StaticBucket ready!")
except ImportError:
    HAS_NATIVE = False
    print("⚠️ Native backend not compiled. Run: python setup.py build_ext --inplace")
    print("   Skipping StaticBucket examples — the high-level API still works with NumPy.")

In [ ]:
if HAS_NATIVE:
    # Create a 5-fold cross-validation mask
    mask = create_fold_mask(n_samples, n_folds=5)

    # Allocate GPU/CPU memory bucket
    bucket = StaticBucket(n_samples, X.shape[1])
    bucket.upload_all(
        [X[:, i] for i in range(X.shape[1])],
        y,
        mask
    )
    print(f"Bucket allocated: {n_samples} samples, {X.shape[1]} features")

    # ----- Compute a single interaction: price * volume -----
    stats = bucket.compute(
        feature_indices=[0, 1],               # f0, f1
        ops=[UnaryOp.IDENTITY, UnaryOp.IDENTITY],  # No transform
        interaction=InteractionType.MULT,       # Multiply
        val_fold=0                             # Validate on fold 0
    )

    train_r, val_r = compute_pearson_from_stats(stats)
    print(f"\nprice × volume: train_r={train_r:.4f}, val_r={val_r:.4f}")
else:
    print("Skipped (no native backend)")

In [ ]:
if HAS_NATIVE:
    # ----- Operator exploration: try all transforms on a pair -----
    print("\n🔍 Operator Search: volatility op1 × momentum op2\n")
    print(f"{'Op1':<12} {'Op2':<12} {'Interaction':<8} {'Train r':>8} {'Val r':>8}")
    print("-" * 55)

    ops = [
        (UnaryOp.IDENTITY, "identity"),
        (UnaryOp.LOG,      "log"),
        (UnaryOp.SQRT,     "sqrt"),
        (UnaryOp.SQUARE,   "square"),
        (UnaryOp.ABS,      "abs"),
        (UnaryOp.TANH,     "tanh"),
    ]

    interactions = [
        (InteractionType.MULT, "*"),
        (InteractionType.ADD,  "+"),
        (InteractionType.DIV,  "/"),
    ]

    best = {"val_r": 0}
    n_tested = 0
    t0 = time.perf_counter()

    for op1_id, op1_name in ops:
        for op2_id, op2_name in ops:
            for int_id, int_name in interactions:
                stats = bucket.compute(
                    feature_indices=[2, 3],  # volatility, momentum
                    ops=[op1_id, op2_id],
                    interaction=int_id,
                    val_fold=0
                )
                train_r, val_r = compute_pearson_from_stats(stats)
                n_tested += 1

                if abs(val_r) > best["val_r"]:
                    best = {
                        "op1": op1_name, "op2": op2_name,
                        "int": int_name,
                        "train_r": abs(train_r), "val_r": abs(val_r)
                    }

    elapsed = time.perf_counter() - t0
    print(f"\nTested {n_tested} combinations in {elapsed*1000:.1f}ms")
    print(f"Best: {best['op1']}(volatility) {best['int']} {best['op2']}(momentum)")
    print(f"  train_r={best['train_r']:.4f}, val_r={best['val_r']:.4f}")
else:
    print("Skipped (no native backend)")

In [ ]:
if HAS_NATIVE:
    # ----- Full pairwise scan with cross-validation -----
    print("\n📊 Full Pairwise Scan (5-fold CV)\n")

    results = []
    t0 = time.perf_counter()
    n_total = 0

    for i, j in combinations(range(X.shape[1]), 2):
        fold_vals = []
        for fold in range(5):
            stats = bucket.compute([i, j], [UnaryOp.IDENTITY, UnaryOp.IDENTITY],
                                   InteractionType.MULT, fold)
            _, val_r = compute_pearson_from_stats(stats)
            fold_vals.append(abs(val_r))
            n_total += 1
        
        mean_val_r = np.mean(fold_vals)
        std_val_r = np.std(fold_vals)
        results.append((feature_names[i], feature_names[j], mean_val_r, std_val_r))

    elapsed = time.perf_counter() - t0
    results.sort(key=lambda x: x[2], reverse=True)

    print(f"{'Feature A':<12} {'Feature B':<12} {'Mean |val_r|':>12} {'Std':>8}")
    print("-" * 50)
    for a, b, mean_r, std_r in results:
        print(f"{a:<12} {b:<12} {mean_r:>12.4f} {std_r:>8.4f}")

    print(f"\n⚡ {n_total} kernel calls in {elapsed*1000:.1f}ms")
    
    # Cleanup
    del bucket
else:
    print("Skipped (no native backend)")

---
## 5️⃣ Loading Real Data — GafimeStreamer

For real datasets, use `GafimeStreamer` to load CSV/Parquet files with automatic VRAM-aware batching.

> Requires `polars` (`pip install polars`)

In [ ]:
from gafime import GafimeStreamer

# --- Example with a CSV file (uncomment and modify path) ---

# streamer = GafimeStreamer(
#     file_path="your_data.csv",     # Path to your CSV or Parquet
#     y_col="target",                # Name of the target column
# )

# # Stream data in VRAM-sized chunks
# for X_batch, y_batch in streamer.stream_with_target(vram_budget_gb=6.0):
#     engine = GafimeEngine()
#     report = engine.analyze(X_batch, y_batch)
#     print(f"Batch: {X_batch.shape} — Signal: {report.decision.signal_detected}")

print("GafimeStreamer is ready for your real data!")
print("Supported formats: .csv, .parquet")
print("\nJust uncomment the code above and point it at your file.")

---
## 🧠 Available Operators & Interactions

### Unary Operators (applied per-feature before combining)
| ID | Name | Formula |
|---|---|---|
| 0 | `IDENTITY` | `x` |
| 1 | `LOG` | `log(\|x\| + ε)` |
| 2 | `EXP` | `exp(clamp(x))` |
| 3 | `SQRT` | `sqrt(\|x\|)` |
| 4 | `SQUARE` | `x²` |
| 5 | `ABS` | `\|x\|` |
| 6 | `INV` | `1 / (x + ε)` |
| 7 | `NEG` | `-x` |
| 8 | `SIGN` | `sign(x)` |
| 9 | `TANH` | `tanh(x)` |
| 10 | `SIGMOID` | `1 / (1 + e⁻ˣ)` |

### Interaction Types (how features are combined)
| ID | Name | Formula |
|---|---|---|
| 0 | `MULT` | `a × b` |
| 1 | `ADD` | `a + b` |
| 2 | `SUB` | `a - b` |
| 3 | `DIV` | `a / (b + ε)` |
| 4 | `MAX` | `max(a, b)` |
| 5 | `MIN` | `min(a, b)` |

---
## 🏗️ Backend Priority

GAFIME automatically selects the fastest available backend:

```
1. CUDA       — NVIDIA GPUs (fastest, requires nvcc)
2. Metal      — Apple Silicon M1/M2/M3/M4 (macOS only)
3. Core       — C++ with pybind11 (fast CPU)
4. NumPy      — Pure Python fallback (always available)
```

Override with `EngineConfig(backend="cuda")`, `backend="metal"`, `backend="cpu"`, or `backend="numpy"`.

---
## 🎯 Quick Tips

1. **Start simple**: Use default `GafimeEngine()` first
2. **Check the decision**: `report.decision.signal_detected` is your go/no-go
3. **Read the p-values**: Low p-values (< 0.05) mean the signal is real
4. **Check stability**: High std across resamples = unreliable result
5. **Increase budget for more features**: Set `max_combinations_per_k` higher
6. **Try 3-way interactions**: Set `max_comb_size=3` (much more expensive)
7. **Use `.to_dict()`**: Export the report as a dictionary for downstream use

```python
# Export report as a dictionary
report_dict = report.to_dict()
```

---
*Happy feature mining! 🚀*